# AI-Native Compliance Infrastructure — Starter Build
### Phase 0–2: Environment Setup, Data Ingestion, Core RAG Engine

**How to use this notebook:**
1. Upload this file to Kaggle → **Create → New Notebook → File → Import Notebook**
2. On the right sidebar: **Settings → Accelerator → GPU T4 x2** (or P100)
3. Run each cell top to bottom (Shift+Enter)
4. Only uses **public** documents and synthetic data — nothing proprietary, nothing from your employer

**What this notebook builds:**
- A working retrieval engine (RAG) that can answer questions about Medicaid policy documents
- The foundation that Phase 3 (change-detection) and Phase 4 (draft generation) will build on top of

---

## Phase 0 — Environment Setup

We're using small, efficient open models so this runs comfortably on Kaggle's free T4 GPU:
- **Embedding model**: `all-MiniLM-L6-v2` (tiny, fast, great for retrieval)
- **Generation model**: `Qwen2.5-7B-Instruct` in 4-bit quantization (~5GB VRAM, fits easily on a 16GB T4)

Why Qwen2.5-7B: strong instruction-following for its size, handles document Q&A well, and is small enough to stay fast on free-tier hardware. We can upgrade to a bigger model later once you move to a paid GPU (RunPod, etc.)

In [ ]:
# Install dependencies (takes ~2-3 min on first run)
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf
print("Dependencies installed.")

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2, then re-run.")

## Phase 1 — Data Collection (Public Documents Only)

For this starter build, drop a few **public** Medicaid/CMS policy PDFs into Kaggle's `/kaggle/input/` via
**Add Data** in the right sidebar, or paste plain text directly into the `SAMPLE_DOCS` list below to get started
immediately without needing real files yet.

Good public sources to pull from later:
- Medicaid.gov state plan amendments
- CMS provider bulletins (publicly posted)
- Federal Register Medicaid rule updates

For now, this cell includes placeholder sample text so you can run the full pipeline end-to-end today.
Replace with real public policy text as soon as you have it.

In [ ]:
# Placeholder sample documents — REPLACE with real public CMS/Medicaid text
# (Each string simulates one policy document/section)

SAMPLE_DOCS = [
    {
        "id": "policy_prior_auth_v1",
        "title": "Prior Authorization Policy — Outpatient Services (Sample, v1)",
        "text": """Effective January 2026, prior authorization is required for all outpatient
        physical therapy services exceeding 12 visits per calendar year. Providers must submit
        requests at least 5 business days before the 13th visit. Requests submitted without
        supporting clinical documentation will be denied. This policy applies to all managed
        care organizations contracted under the state Medicaid program."""
    },
    {
        "id": "policy_prior_auth_v2",
        "title": "Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)",
        "text": """Effective July 2026, prior authorization is required for all outpatient
        physical therapy services exceeding 20 visits per calendar year, increased from the
        previous threshold of 12 visits. Providers must submit requests at least 3 business
        days before the 21st visit, reduced from the prior 5-day requirement. Requests
        submitted without supporting clinical documentation will be denied. This policy
        applies to all managed care organizations contracted under the state Medicaid program."""
    },
    {
        "id": "policy_telehealth",
        "title": "Telehealth Reimbursement Policy (Sample)",
        "text": """Telehealth visits are reimbursed at parity with in-person visits for
        behavioral health services only. Audio-only visits are reimbursed at 80% of the
        standard rate. Providers must document the modality of service delivery on every claim."""
    },
]

print(f"Loaded {len(SAMPLE_DOCS)} sample documents.")
print("Replace SAMPLE_DOCS with real public policy text when ready.")

In [ ]:
# Optional: load real PDFs if you've added them via Kaggle "Add Data"
import os
from pypdf import PdfReader

def load_pdfs_from_folder(folder_path):
    """Reads every PDF in a folder and returns them in the same format as SAMPLE_DOCS."""
    docs = []
    if not os.path.exists(folder_path):
        print(f"No folder found at {folder_path} — skipping. Using SAMPLE_DOCS instead.")
        return docs
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(".pdf"):
            reader = PdfReader(os.path.join(folder_path, fname))
            text = "\n".join(page.extract_text() or "" for page in reader.pages)
            docs.append({"id": fname, "title": fname, "text": text})
    return docs

# Example: uncomment once you've added a Kaggle dataset with real public PDFs
# real_docs = load_pdfs_from_folder("/kaggle/input/your-dataset-name")
# if real_docs:
#     SAMPLE_DOCS = real_docs
#     print(f"Loaded {len(SAMPLE_DOCS)} real documents instead.")

## Phase 2 — Core RAG Engine

### Step 1: Chunk documents
Long documents need to be split into smaller passages so retrieval is precise.

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Simple word-based chunking with overlap so context isn't lost at boundaries."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

# Build the chunk store
all_chunks = []
for doc in SAMPLE_DOCS:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "chunk_id": f'{doc["id"]}_{i}',
            "text": chunk,
        })

print(f"Created {len(all_chunks)} chunks from {len(SAMPLE_DOCS)} documents.")

### Step 2: Embed chunks and build a searchable index
This turns each text chunk into a vector, so we can find the most relevant chunks for any question.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Index built with {index.ntotal} vectors of dimension {dimension}.")

### Step 3: Retrieval function
Given a question, find the most relevant chunks.

In [ ]:
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = [all_chunks[i] for i in indices[0]]
    return results

# Quick test — no LLM needed yet, just checking retrieval quality
test_results = retrieve("What is the prior authorization threshold for physical therapy?")
for r in test_results:
    print(f"[{r['title']}]")
    print(r["text"][:200], "...\n")

### Step 4: Load the generation model (Qwen2.5-7B-Instruct, 4-bit)
This is the model that will read the retrieved chunks and write a plain-language answer.
First load will download ~5GB — takes a few minutes on Kaggle.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")

### Step 5: Generation function — combine retrieval + LLM
This is the core of the RAG engine: retrieve relevant policy text, then ask the model to answer
using ONLY that retrieved context (reduces hallucination, keeps answers grounded in real source text).

In [ ]:
def ask(query, top_k=3, max_new_tokens=300):
    retrieved = retrieve(query, top_k=top_k)
    context = "\n\n".join(f"[Source: {r['title']}]\n{r['text']}" for r in retrieved)

    prompt = f"""You are a Medicaid compliance assistant. Answer the question using ONLY the
context provided below. If the context does not contain the answer, say so clearly.
Always cite which source document your answer comes from.

Context:
{context}

Question: {query}

Answer:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response, retrieved

# Test it
answer, sources = ask("What is the current prior authorization threshold for outpatient physical therapy?")
print("ANSWER:\n", answer)
print("\nSOURCES USED:", [s["title"] for s in sources])

---
## What You've Built So Far

You now have a working RAG engine that:
- Ingests policy documents (sample or real public PDFs)
- Retrieves the most relevant passages for any question
- Generates grounded, cited answers using an open-source 7B model
- Runs entirely on Kaggle's free GPU — no local hardware needed

## Try It Yourself
Run the cell below with your own questions to see the engine in action.

In [ ]:
# Try your own question
my_question = "What changed about telehealth reimbursement?"
answer, sources = ask(my_question)
print("ANSWER:\n", answer)
print("\nSOURCES:", [s["title"] for s in sources])

---
## Next Steps (Phase 3 — coming next)

Notice the two sample documents `policy_prior_auth_v1` and `policy_prior_auth_v2` above — they're
the same policy at two points in time. **Phase 3 builds the "diff" layer**: automatically detecting
that v2 changed the visit threshold from 12→20 and the notice period from 5→3 days, then mapping
that change to which internal SOPs it would affect.

That's the feature that makes this a *compliance* product instead of just a chatbot — save this
notebook (File → Save Version on Kaggle) and we'll build that layer next.

## **Phase 3 — The Change-Detection ("Diff") Layer**

**How to use this**: copy each cell below into your existing notebook, right after the "What You've Built So Far" section (before "Next Steps"). Run them in order.

This is the layer that turns your RAG engine from a Q&A chatbot into an actual **compliance tool**: it explicitly tracks which policy version is current, detects what changed between versions, and forces every answer to use the current version — while still being able to explain what changed from the old one.

In [26]:
# Add version metadata to the sample docs
# In a real system this would come from document dates / version numbers in the source PDFs

DOC_METADATA = {
    "policy_prior_auth_v1": {
        "policy_family": "prior_auth_outpatient_pt",
        "version": 1,
        "effective_date": "2026-01-01",
        "is_current": False,
    },
    "policy_prior_auth_v2": {
        "policy_family": "prior_auth_outpatient_pt",
        "version": 2,
        "effective_date": "2026-07-01",
        "is_current": True,
    },
    "policy_telehealth": {
        "policy_family": "telehealth_reimbursement",
        "version": 1,
        "effective_date": "2026-01-01",
        "is_current": True,
    },
}

# Attach metadata to each doc
for doc in SAMPLE_DOCS:
    doc["metadata"] = DOC_METADATA.get(doc["id"], {
        "policy_family": doc["id"], "version": 1,
        "effective_date": "unknown", "is_current": True,
    })

print("Metadata attached. Current documents:")
for doc in SAMPLE_DOCS:
    if doc["metadata"]["is_current"]:
        print(f"  - {doc['title']}")

Metadata attached. Current documents:
  - Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)
  - Telehealth Reimbursement Policy (Sample)


## Step 2 — Detect changes between versions

For any policy family with more than one version, compare the old and new text and ask the
LLM to summarize exactly what changed. This is the core "diff" feature.

In [27]:
def find_version_pairs():
    """Groups documents by policy family and returns (old, new) pairs where >1 version exists."""
    families = {}
    for doc in SAMPLE_DOCS:
        fam = doc["metadata"]["policy_family"]
        families.setdefault(fam, []).append(doc)

    pairs = []
    for fam, docs in families.items():
        if len(docs) > 1:
            docs_sorted = sorted(docs, key=lambda d: d["metadata"]["version"])
            pairs.append((docs_sorted[-2], docs_sorted[-1]))  # previous vs current
    return pairs

pairs = find_version_pairs()
print(f"Found {len(pairs)} policy update(s) to analyze.")
for old, new in pairs:
    print(f"  - {old['title']}  -->  {new['title']}")

Found 1 policy update(s) to analyze.
  - Prior Authorization Policy — Outpatient Services (Sample, v1)  -->  Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)


In [28]:
def detect_changes(old_doc, new_doc, max_new_tokens=250):
    """Asks the LLM to summarize exactly what changed between two versions of a policy."""
    prompt = f"""You are a Medicaid compliance analyst. Compare these two versions of the
same policy and list ONLY the concrete changes (numbers, deadlines, thresholds, wording that
changes an obligation). Be specific and use a short bullet list. If nothing substantive
changed, say so.

OLD VERSION ({old_doc['metadata']['effective_date']}):
{old_doc['text']}

NEW VERSION ({new_doc['metadata']['effective_date']}):
{new_doc['text']}

Changes:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True,
        )

    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Run change detection on every version pair found
change_reports = []
for old, new in pairs:
    print(f"Analyzing: {new['title']}...")
    changes = detect_changes(old, new)
    change_reports.append({
        "policy_family": new["metadata"]["policy_family"],
        "old_doc": old["title"],
        "new_doc": new["title"],
        "changes": changes,
    })
    print(changes)
    print("-" * 60)

Analyzing: Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)...
- **Threshold for requiring prior authorization:** Increased from 12 visits to 20 visits.
- **Deadline for submitting prior authorization requests:** Reduced from 5 business days to 3 business days.
- **Number of visits triggering prior authorization:** Increased from the 13th visit to the 21st visit.
------------------------------------------------------------


## Step 3 — Map changes to affected internal SOPs

In a real deployment, a client would upload their internal SOPs/procedures. Here we simulate
that with a small sample set of "internal procedures" and have the model flag which ones are
affected by each detected change.

In [29]:
# Simulated internal client SOPs (in a real deployment these come from the client's own docs)
SAMPLE_INTERNAL_SOPS = [
    {"id": "sop_001", "title": "Outpatient PT Intake Procedure",
     "text": "Front desk staff verify visit count against the 12-visit prior authorization "
             "threshold before scheduling. If a patient exceeds 12 visits, flag for prior auth."},
    {"id": "sop_002", "title": "Claims Denial Appeals Process",
     "text": "Appeals for denied physical therapy claims must reference the prior authorization "
             "policy visit threshold and the 5-business-day submission window."},
    {"id": "sop_003", "title": "Telehealth Billing Checklist",
     "text": "Confirm modality (audio-only vs video) is documented on every telehealth claim "
             "before submission."},
]

def find_affected_sops(change_summary, sops, max_new_tokens=200):
    """Asks the LLM which internal SOPs are affected by a given policy change."""
    sop_list = "\n\n".join(f"[{s['id']}] {s['title']}: {s['text']}" for s in sops)
    prompt = f"""Given this policy change:
{change_summary}

And these internal procedures:
{sop_list}

Which procedure(s) are affected by this change, and why? List by ID. If none are affected, say so."""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Run impact mapping for each detected change
for report in change_reports:
    print(f"=== Impact analysis: {report['new_doc']} ===")
    impact = find_affected_sops(report["changes"], SAMPLE_INTERNAL_SOPS)
    report["affected_sops"] = impact
    print(impact)
    print("-" * 60)

=== Impact analysis: Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED) ===
The policy changes affect the following procedures:

- **[sop_001] Outpatient PT Intake Procedure**: This procedure is directly impacted because the threshold for requiring prior authorization has increased from 12 visits to 20 visits. The front desk staff will now need to flag for prior authorization starting at the 21st visit instead of the 13th visit.

- **[sop_002] Claims Denial Appeals Process**: This procedure is also affected because the prior authorization policy visit threshold has changed. The appeals process must now reference the new 20-visit threshold instead of the previous 12-visit threshold.

The deadline for submitting prior authorization requests does not affect [sop_002] since it pertains to the time frame for submission rather than the number of visits triggering prior authorization.

- **[sop_003] Telehealth Billing Checklist**: This procedure is not affected by the pol

## Step 4 — Generate the full compliance report

This ties Phase 3 together: change detected → impact mapped → a clean, structured report a
compliance analyst could actually act on.

In [31]:
def generate_compliance_report(report):
    """Formats a single change report into a clean, readable summary."""
    output = f"""
{'=' * 70}
COMPLIANCE ALERT — {report['policy_family']}
{'=' * 70}

Previous version: {report['old_doc']}
Current version:  {report['new_doc']}

WHAT CHANGED:
{report['changes']}

AFFECTED INTERNAL PROCEDURES:
{report['affected_sops']}

STATUS: Pending review — draft SOP updates not yet generated
{'=' * 70}
"""
    return output

for report in change_reports:
    print(generate_compliance_report(report))


COMPLIANCE ALERT — prior_auth_outpatient_pt

Previous version: Prior Authorization Policy — Outpatient Services (Sample, v1)
Current version:  Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)

WHAT CHANGED:
- **Threshold for requiring prior authorization:** Increased from 12 visits to 20 visits.
- **Deadline for submitting prior authorization requests:** Reduced from 5 business days to 3 business days.
- **Number of visits triggering prior authorization:** Increased from the 13th visit to the 21st visit.

AFFECTED INTERNAL PROCEDURES:
The policy changes affect the following procedures:

- **[sop_001] Outpatient PT Intake Procedure**: This procedure is directly impacted because the threshold for requiring prior authorization has increased from 12 visits to 20 visits. The front desk staff will now need to flag for prior authorization starting at the 21st visit instead of the 13th visit.

- **[sop_002] Claims Denial Appeals Process**: This procedure is also affecte

---
## What Phase 3 Adds

Your engine now:
- Knows which policy version is current vs outdated
- Automatically detects and summarizes what changed between versions
- Maps changes to specific affected internal procedures
- Produces a structured compliance report, not just a raw answer

This is the demo-ready core: point it at real public CMS policy updates and a sample set of
internal SOPs, and it produces exactly the "detect impact → draft update" flow we scoped for
your agency/investor demo.

## Next: Phase 4 — Draft Generation
The next step is generating an actual **draft SOP update** (not just flagging what's affected),
with citations back to the source policy — the piece that makes this a finished demo flow.

# Phase 4 — Draft Generation (+ Phase 5 prep)

**How to use this:** copy each cell below into your notebook, right after the Phase 3 cells
(after "What Phase 3 Adds", before "Next: Phase 4"). Run in order.

This phase does two things:
1. **Generates actual draft SOP update language** — not just "this is affected," but the
   rewritten paragraph, with citations back to the source policy.
2. **Packages everything into one clean function and a JSON export** — this is what Phase 5
   (the UI) will call. Building it this way now means the UI step is just "call one function
   and display the result," instead of re-wiring your pipeline later.

## Step 1 — Generate a draft SOP update

Given a detected change and the specific SOP it affects, ask the model to write the actual
updated procedure language, citing the new policy.

In [52]:
def generate_sop_draft(sop, change_summary, new_doc, max_new_tokens=300):
    """Generates a draft rewrite of one internal SOP, reflecting a detected policy change."""
    prompt = f"""You are a Medicaid compliance analyst drafting an SOP update. Rewrite the
procedure below to reflect the policy change. Keep the same structure and tone as the
original, change only what the policy change requires, and end with a citation line noting
the source policy and effective date.

ORIGINAL PROCEDURE [{sop['id']}] {sop['title']}:
{sop['text']}

POLICY CHANGE:
{change_summary}

SOURCE POLICY: {new_doc['title']} (effective {new_doc['metadata']['effective_date']})

Rewritten procedure:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print("generate_sop_draft() ready.")

generate_sop_draft() ready.


## Step 2 — Decide which SOPs need a drafted update

Not every affected SOP needs a rewrite (e.g. one might just need a reference number updated
elsewhere). We ask the model to classify each affected SOP as "needs rewrite" or "no rewrite
needed," then only draft the ones that do — this keeps the demo output focused and avoids
wasted generation time.

In [53]:
import re

def parse_affected_sop_ids(impact_text, sops):
    """Extracts which SOP IDs were flagged as affected from the impact analysis text."""
    affected_ids = []
    for sop in sops:
        if sop["id"] in impact_text:
            affected_ids.append(sop["id"])
    return affected_ids


def needs_rewrite(sop, change_summary, impact_text):
    """Simple heuristic: if the SOP ID appears with 'affected' language near it, draft it.
    (For the demo, we draft every flagged SOP — a production version could add a
    classification step here to skip trivial/no-op changes.)"""
    return sop["id"] in impact_text


print("Helper functions ready.")

Helper functions ready.


## Step 3 — One function that runs the whole pipeline

This is the important part for Phase 5: instead of scattered cells, everything (change
detection -> impact mapping -> draft generation) is wrapped into a single function that
takes a policy family and returns one clean, structured result. **This is the function the
UI will call.**

In [54]:
def run_full_compliance_check(old_doc, new_doc, sops):
    """
    Runs the entire Phase 3 + Phase 4 pipeline for one policy update and returns a single
    structured dict — this is the function Phase 5's UI will call directly.
    """
    # Step A: detect changes
    changes = detect_changes(old_doc, new_doc)

    # Step B: map impact
    impact_text = find_affected_sops(changes, sops)
    affected_ids = parse_affected_sop_ids(impact_text, sops)

    # Step C: draft updates for affected SOPs
    drafts = []
    for sop in sops:
        if sop["id"] in affected_ids:
            draft_text = generate_sop_draft(sop, changes, new_doc)
            drafts.append({
                "sop_id": sop["id"],
                "sop_title": sop["title"],
                "original_text": sop["text"],
                "draft_text": draft_text,
            })

    result = {
        "policy_family": new_doc["metadata"]["policy_family"],
        "old_version": {
            "title": old_doc["title"],
            "effective_date": old_doc["metadata"]["effective_date"],
        },
        "new_version": {
            "title": new_doc["title"],
            "effective_date": new_doc["metadata"]["effective_date"],
        },
        "changes_summary": changes,
        "impact_analysis": impact_text,
        "affected_sop_ids": affected_ids,
        "drafted_updates": drafts,
        "status": "drafts_ready_for_review",
    }
    return result


print("run_full_compliance_check() ready — this is the main entry point for Phase 5.")

run_full_compliance_check() ready — this is the main entry point for Phase 5.


## Step 4 — Run the full pipeline end to end

This replaces the separate Phase 3 loop — one call now produces everything, including drafts.

In [55]:
# Run the complete pipeline for every detected policy update
full_results = []
for old, new in pairs:
    print(f"Running full pipeline for: {new['title']}...")
    result = run_full_compliance_check(old, new, SAMPLE_INTERNAL_SOPS)
    full_results.append(result)
    print(f"  -> {len(result['drafted_updates'])} draft update(s) generated.")

print("\nDone. Full results stored in `full_results`.")

Running full pipeline for: Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)...
  -> 3 draft update(s) generated.

Done. Full results stored in `full_results`.


# Step 5 - Print a demo-ready view
A clean, readable version of the final output — this is what you'd literally show on screen during a demo.

In [56]:
def print_full_report(result):
    print("=" * 70)
    print(f"COMPLIANCE UPDATE — {result['policy_family']}")
    print("=" * 70)
    print(f"\n{result['old_version']['title']}  -->  {result['new_version']['title']}")
    print(f"\nWHAT CHANGED:\n{result['changes_summary']}")
    print(f"\nIMPACT ANALYSIS:\n{result['impact_analysis']}")

    for draft in result["drafted_updates"]:
        print(f"\n{'-' * 70}")
        print(f"DRAFT UPDATE — [{draft['sop_id']}] {draft['sop_title']}")
        print(f"{'-' * 70}")
        print(f"\nORIGINAL:\n{draft['original_text']}")
        print(f"\nDRAFT REWRITE:\n{draft['draft_text']}")

    print(f"\nSTATUS: {result['status']}")
    print("=" * 70)


for result in full_results:
    print_full_report(result)

COMPLIANCE UPDATE — prior_auth_outpatient_pt

Prior Authorization Policy — Outpatient Services (Sample, v1)  -->  Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)

WHAT CHANGED:
- **Threshold for Prior Authorization:** Increased from 12 visits to 20 visits.
- **Deadline for Submission:** Reduced from 5 business days to 3 business days.
- **Visit Count for Submission:** Changed from "before the 13th visit" to "before the 21st visit."

IMPACT ANALYSIS:
The policy changes affect the following procedures:

- **[sop_001] Outpatient PT Intake Procedure**: This procedure involves verifying the visit count against the prior authorization threshold. With the new threshold increased to 20 visits, the front desk staff will need to update their process to reflect this change. They should now flag for prior authorization if a patient has reached or exceeded 20 visits instead of 12.

- **[sop_002] Claims Denial Appeals Process**: The appeals process needs to be updated to refl

## Step 6 — Export to JSON (this is what Phase 5 needs)

Phase 5 will build a UI that displays this data. Rather than the UI needing to talk to this
notebook directly, we export the structured results to a JSON file. This becomes realistic
sample data to build the UI against immediately, and later can be swapped for a live call to
`run_full_compliance_check()` if you deploy this as a real backend.

In [57]:
import json

with open("/kaggle/working/compliance_results.json", "w") as f:
    json.dump(full_results, f, indent=2)

print("Saved to /kaggle/working/compliance_results.json")
print(f"Contains {len(full_results)} compliance update record(s).")

# Preview the structure
print("\nPreview of one record's keys:")
print(list(full_results[0].keys()))

Saved to /kaggle/working/compliance_results.json
Contains 1 compliance update record(s).

Preview of one record's keys:
['policy_family', 'old_version', 'new_version', 'changes_summary', 'impact_analysis', 'affected_sop_ids', 'drafted_updates', 'status']


---
## What Phase 4 Adds

Your engine now produces the complete flow end to end:
**detect change -> map impact -> draft the actual updated procedure -> package as structured
data.**

This is the full "detect impact -> draft update" demo flow originally scoped for your
agency/investor pitch — nothing left to fake or hand-wave.

## Ready for Phase 5

Two things are now in place specifically to make the UI build easy:
1. **`run_full_compliance_check()`** — one function, clean inputs/outputs, ready to be called
   from a backend if this ever goes beyond a local demo.
2. **`compliance_results.json`** — download this file from `/kaggle/working/`. It's realistic
   structured data we can load directly into a polished web UI, so Phase 5 can start
   immediately without waiting on live model calls every time you tweak the design.

**Next step:** download `compliance_results.json` from your Kaggle output panel, and share it
back — that's what I'll use to build the Phase 5 front-end.